# LeetCode 275 — H-Index II bằng Python

Link đề: https://leetcode.com/problems/h-index-ii/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search

Skeleton LeetCode:

```python
class Solution:
    def hIndex(self, citations: List[int]) -> int:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ đề gốc; mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với một lời giải brute force độc lập (thử mọi `h` và đếm trực tiếp số bài có ít nhất `h` trích dẫn) trên mảng đã sắp tăng ngẫu nhiên.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: giống bài H-Index (274) nhưng mảng `citations` **đã được sắp tăng dần**, và ta phải giải trong thời gian **logarit** `O(log n)`; h-index là `h` **lớn nhất** sao cho có **ít nhất `h` bài** mà **mỗi bài** có **ít nhất `h` trích dẫn**.
2. Nhận ra khi mảng đã sắp tăng, "số bài có `≥ h` trích dẫn" là **`h` phần tử cuối**, và `h` hợp lệ **khi và chỉ khi `citations[n − h] ≥ h`** (phần tử nhỏ nhất trong `h` bài cuối vẫn đủ `h` trích dẫn).
3. Thấy **tính đơn điệu** của điều kiện (hợp lệ cho `h = 0..H`, rồi không hợp lệ) và đi từ **quét tuyến tính** → **tìm nhị phân trên `h` (cách tham khảo)** → **tìm nhị phân trên chỉ số (lower bound)** → **`bisect` với tham số `key`** → **so sánh với cách đếm thùng ở bài 274**.
4. Nắm vững **khuôn tìm nhị phân "cận dưới / cận trên"** (`lo`, `hi`, `mid`, khi nào dùng `(lo + hi + 1) // 2`) và cách đổi giữa chỉ số `i` và giá trị `h = n − i`.
5. Tránh các lỗi kinh điển: vòng lặp vô hạn do `mid` không làm hẹp khoảng, trả về chỉ số thay vì `h`, lệch chỉ số `n − h` và `n − h − 1`, điều kiện `>` thay vì `>=`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `citations` trong đó `citations[i]` là số trích dẫn của bài báo thứ `i` của một nhà nghiên cứu và `citations` được **sắp theo thứ tự không giảm**. Hãy trả về **h-index** của nhà nghiên cứu.

Theo định nghĩa của Wikipedia: h-index là giá trị `h` **lớn nhất** sao cho nhà nghiên cứu đã công bố **ít nhất `h` bài** mà **mỗi bài** được trích dẫn **ít nhất `h` lần**.

Bạn phải viết thuật toán chạy trong thời gian **logarit**.

**Ví dụ 1**

- Input: `citations = [0,1,3,5,6]`
- Output: `3`
- Giải thích: có `5` bài với `0, 1, 3, 5, 6` trích dẫn. Có `3` bài được trích dẫn ít nhất `3` lần (`3, 5, 6`) và hai bài còn lại không quá `3`. Vậy h-index là `3`.

**Ví dụ 2**

- Input: `citations = [1,2,100]`
- Output: `2`

**Ràng buộc:** `n == citations.length`; `1 <= n <= 10⁵`; `0 <= citations[i] <= 1000`; `citations` được sắp tăng dần.

In [1]:
from typing import List
import random
from bisect import bisect_left

examples = [
    ([0, 1, 3, 5, 6], 3),
    ([1, 2, 100], 2),
]
for citations, expected in examples:
    print(f"citations={citations}, kỳ vọng {expected}")

citations=[0, 1, 3, 5, 6], kỳ vọng 3
citations=[1, 2, 100], kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Mảng đã sắp tăng: `h` bài cuối là `h` bài nhiều trích dẫn nhất

Vì mảng sắp **tăng**, `h` phần tử **cuối** (chỉ số `n − h ... n − 1`) là `h` bài được trích dẫn nhiều nhất. Phần tử **nhỏ nhất** trong số đó là `citations[n − h]`. Nếu phần tử nhỏ nhất đó vẫn `≥ h` thì cả `h` bài đều `≥ h`: `h` hợp lệ.

In [2]:
c = [0, 1, 3, 5, 6]
n = len(c)
for h in range(0, n + 1):
    last = c[n - h:]                             # h bài cuối
    ok = (h == 0) or c[n - h] >= h
    print(f"h={h}: h bài cuối = {last}, nhỏ nhất = {c[n - h] if h else '-'}, hợp lệ? {ok}")

h=0: h bài cuối = [], nhỏ nhất = -, hợp lệ? True
h=1: h bài cuối = [6], nhỏ nhất = 6, hợp lệ? True
h=2: h bài cuối = [5, 6], nhỏ nhất = 5, hợp lệ? True
h=3: h bài cuối = [3, 5, 6], nhỏ nhất = 3, hợp lệ? True
h=4: h bài cuối = [1, 3, 5, 6], nhỏ nhất = 1, hợp lệ? False
h=5: h bài cuối = [0, 1, 3, 5, 6], nhỏ nhất = 0, hợp lệ? False


### 3.2. Điều kiện đơn điệu: `True ... True False ... False`

Điều kiện "`h` hợp lệ" đúng cho `h = 0, 1, ..., H` và sai từ `H + 1` trở đi. Đáp án `H` là phần tử `True` **cuối cùng**. Bài toán dạng "tìm điểm chuyển của một điều kiện đơn điệu" giải bằng tìm nhị phân.

In [3]:
c = [0, 1, 3, 5, 6]
n = len(c)
valid = [h == 0 or c[n - h] >= h for h in range(n + 1)]
print(valid)                                     # [True, True, True, True, False, False]
assert valid == sorted(valid, reverse=True)

[True, True, True, True, False, False]


### 3.3. Hai khuôn tìm nhị phân

Cho điều kiện đơn điệu `P(x)`:

- **Tìm `True` cuối cùng** (khoảng `[lo, hi]`, `P(lo)` đúng): `while lo < hi: mid = (lo + hi + 1) // 2; if P(mid): lo = mid else: hi = mid − 1`. Dùng `+ 1` để `mid > lo`, tránh lặp vô hạn khi `lo = mid`.
- **Tìm `True` đầu tiên** (`P(hi)` đúng): `while lo < hi: mid = (lo + hi) // 2; if P(mid): hi = mid else: lo = mid + 1`. Dùng `mid` làm tròn xuống để `mid < hi`.

Cả hai thu hẹp khoảng sau mỗi bước, nên dừng sau `O(log n)` bước.

In [4]:
def last_true(lo, hi, pred):
    while lo < hi:
        mid = (lo + hi + 1) // 2        # làm tròn LÊN
        if pred(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo


def first_true(lo, hi, pred):
    while lo < hi:
        mid = (lo + hi) // 2            # làm tròn XUỐNG
        if pred(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


print(last_true(0, 10, lambda x: x <= 6))      # 6
print(first_true(0, 10, lambda x: x >= 4))     # 4

6
4


### 3.4. Chuyển đổi giữa chỉ số `i` và `h = n − i`

Gọi `i` là chỉ số đầu của `h` bài cuối: `i = n − h`. Điều kiện `citations[n − h] ≥ h` viết theo `i` là `citations[i] ≥ n − i`. Khi `h` **tăng**, `i` **giảm**. Vì vậy "`h` lớn nhất hợp lệ" tương ứng với "`i` **nhỏ nhất** thỏa `citations[i] ≥ n − i`" (cận dưới theo `i`).

In [5]:
c = [0, 1, 3, 5, 6]
n = len(c)
for i in range(n):
    print(f"i={i}: c[i]={c[i]} >= n - i = {n - i}? {c[i] >= n - i}  (h = {n - i})")

i=0: c[i]=0 >= n - i = 5? False  (h = 5)
i=1: c[i]=1 >= n - i = 4? False  (h = 4)
i=2: c[i]=3 >= n - i = 3? True  (h = 3)
i=3: c[i]=5 >= n - i = 2? True  (h = 2)
i=4: c[i]=6 >= n - i = 1? True  (h = 1)


### 3.5. `bisect_left` với tham số `key` (Python 3.10+)

`bisect_left(a, x, key=f)` tìm vị trí chèn `x` vào dãy `a` **theo giá trị `f(phần tử)`** mà dãy `f(a[i])` phải **không giảm**. Ta có thể dùng nó để tìm `True` đầu tiên của một điều kiện đơn điệu: `bisect_left(range(n), True, key=lambda i: c[i] >= n - i)` (vì `False < True`, dãy `False…False True…True` là không giảm).

In [6]:
c = [0, 1, 3, 5, 6]
n = len(c)
i = bisect_left(range(n), True, key=lambda i: c[i] >= n - i)
print("chỉ số i đầu tiên thỏa:", i, "-> h =", n - i)
print(list(map(lambda i: c[i] >= n - i, range(n))))        # [False, False, True, True, True]

chỉ số i đầu tiên thỏa: 2 -> h = 3
[False, False, True, True, True]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ `[0, 1, 3, 5, 6]`, `n = 5`. Lập bảng theo chỉ số `i` (số bài từ `i` đến cuối là `n − i`):

| `i` | `c[i]` | `n − i` (số bài từ `i`) | `c[i] ≥ n − i`? |
|---|---|---|---|
| 0 | 0 | 5 | không |
| 1 | 1 | 4 | không |
| 2 | 3 | 3 | **có** |
| 3 | 5 | 2 | có |
| 4 | 6 | 1 | có |

Cột cuối là `False, False, True, True, True`: **đơn điệu**. `i` đầu tiên thỏa là `2`, nên `h = n − 2 = 3`. Lý do: từ `i = 2` trở đi có `3` bài, bài ít nhất có `3` trích dẫn `≥ 3`; còn nếu lấy thêm bài `i = 1` (4 bài) thì bài ít nhất chỉ có `1 < 4`.

Khi mảng đã sắp, ta có **dãy điều kiện đơn điệu** nên không cần quét hết: tìm nhị phân trong `O(log n)`.

In [7]:
c = [1, 2, 100]
n = len(c)
print([(i, c[i], n - i, c[i] >= n - i) for i in range(n)])
i = next((i for i in range(n) if c[i] >= n - i), n)
print("i đầu tiên thỏa:", i, "-> h =", n - i)           # i = 1 -> h = 2

[(0, 1, 3, False), (1, 2, 2, True), (2, 100, 1, True)]
i đầu tiên thỏa: 1 -> h = 2


## 5. Cách 1 — Brute Force / baseline: quét từ trái sang phải (`O(n)`)

**Ý tưởng.** Quét `i = 0, 1, ...`; `i` **đầu tiên** có `c[i] ≥ n − i` cho `h = n − i` (lớn nhất vì `i` nhỏ nhất). Nếu không có `i` nào thì `h = 0`. Đúng nhưng `O(n)`, chưa đạt yêu cầu logarit.

In [8]:
def hindex2_linear(citations: list[int]) -> int:
    n = len(citations)
    for i in range(n):
        if citations[i] >= n - i:               # n - i bài từ i đến cuối đều >= citations[i] >= n - i
            return n - i
    return 0


for citations, expected in examples:
    print(f"citations={citations} -> {hindex2_linear(citations)} (kỳ vọng {expected})")

citations=[0, 1, 3, 5, 6] -> 3 (kỳ vọng 3)
citations=[1, 2, 100] -> 2 (kỳ vọng 2)


**Độ phức tạp (Cách 1).** **Time `O(n)`**, **Space `O(1)`**. Với `n = 10⁵` vẫn nhanh nhưng đề yêu cầu **logarit**.

**Nút thắt.** Điều kiện `c[i] ≥ n − i` là `False…False True…True`: ta không cần xét từng `i`.

## 6. Ý tưởng thuật toán / bước đột phá

**Tìm nhị phân trên đáp án.** Đáp án `h ∈ [0, n]`. Điều kiện `P(h) = (h == 0) or citations[n − h] ≥ h` là **`True…True False…False`** theo `h`:

- nếu `P(h)` đúng với `h` thì đúng với mọi `h' < h` (các bài cuối ít hơn và cần ít trích dẫn hơn);
- `P(0)` luôn đúng.

Dùng khuôn "tìm `True` cuối cùng": `lo = 0`, `hi = n`, `mid = (lo + hi + 1) // 2`; nếu `P(mid)` thì `lo = mid` ngược lại `hi = mid − 1`. Kết quả `lo` là h-index sau `O(log n)` bước (mỗi bước chỉ đọc **một** phần tử `citations[n − mid]`).

**Kiểm tra đơn điệu của `P`:** nếu `citations[n − h] ≥ h` thì với `h' < h`: `n − h' > n − h` nên `citations[n − h'] ≥ citations[n − h] ≥ h > h'`: đúng. Do đó khoảng `True` là một đoạn đầu.

In [9]:
c = [0, 1, 3, 5, 6]
n = len(c)
lo, hi = 0, n
step = 0
while lo < hi:
    step += 1
    mid = (lo + hi + 1) // 2
    ok = c[n - mid] >= mid
    print(f"bước {step}: lo={lo}, hi={hi}, mid={mid}, c[n-mid]=c[{n - mid}]={c[n - mid]} {'>=' if ok else '<'} {mid}")
    if ok:
        lo = mid
    else:
        hi = mid - 1
print("h-index =", lo)

bước 1: lo=0, hi=5, mid=3, c[n-mid]=c[2]=3 >= 3
bước 2: lo=3, hi=5, mid=4, c[n-mid]=c[1]=1 < 4
h-index = 3


## 7. Các cách tiếp theo

### Cách 2 — Tìm nhị phân trên `h` (cách của lời giải tham khảo)

**Giải thích.** Đúng như mục 6: `left, right = 0, n`; `mid = (left + right + 1) >> 1`; nếu `citations[n − mid] ≥ mid` thì `left = mid` ngược lại `right = mid − 1`. Chú ý `mid ≥ 1` trong vòng lặp (vì `left < right` nên `mid ≥ left + 1 ≥ 1`), nên `n − mid` không bị chỉ số âm.

In [10]:
def hindex2_binary_h(citations: list[int]) -> int:
    n = len(citations)
    left, right = 0, n
    while left < right:
        mid = (left + right + 1) >> 1               # làm tròn lên: mid >= left + 1 >= 1
        if citations[n - mid] >= mid:                # h = mid hợp lệ
            left = mid                               # thử h lớn hơn
        else:
            right = mid - 1                          # h = mid không hợp lệ
    return left


for citations, expected in examples:
    print(f"citations={citations} -> {hindex2_binary_h(citations)} (kỳ vọng {expected})")

citations=[0, 1, 3, 5, 6] -> 3 (kỳ vọng 3)
citations=[1, 2, 100] -> 2 (kỳ vọng 2)


In [11]:
# Code nộp LeetCode cho Cách 2
from typing import List


class Solution:
    def hIndex(self, citations: List[int]) -> int:
        n = len(citations)
        left, right = 0, n
        while left < right:
            mid = (left + right + 1) >> 1
            if citations[n - mid] >= mid:
                left = mid
            else:
                right = mid - 1
        return left

**Độ phức tạp Cách 2:** **Time `O(log n)`** (khoảng `[0, n]` giảm một nửa mỗi bước), **Space `O(1)`**.

### Cách 3 — Tìm nhị phân trên chỉ số (cận dưới), `h = n − i`

**Giải thích.** Tìm **`i` nhỏ nhất** sao cho `citations[i] ≥ n − i` với `lo = 0`, `hi = n` (nếu không có `i` nào thỏa ta coi `i = n`, `h = 0`). `mid = (lo + hi) // 2`; nếu `mid < n` và `citations[mid] ≥ n − mid` thì `hi = mid` ngược lại `lo = mid + 1`. Đáp án `n − lo`.

In [12]:
def hindex2_binary_index(citations: list[int]) -> int:
    n = len(citations)
    lo, hi = 0, n                                   # hi = n nghĩa là "không có i nào thỏa" (h = 0)
    while lo < hi:
        mid = (lo + hi) // 2                        # làm tròn xuống: mid < hi
        if citations[mid] >= n - mid:               # i = mid thỏa: tìm i nhỏ hơn
            hi = mid
        else:
            lo = mid + 1
    return n - lo


for citations, expected in examples:
    print(f"citations={citations} -> {hindex2_binary_index(citations)} (kỳ vọng {expected})")

citations=[0, 1, 3, 5, 6] -> 3 (kỳ vọng 3)
citations=[1, 2, 100] -> 2 (kỳ vọng 2)


**Độ phức tạp Cách 3:** **Time `O(log n)`**, **Space `O(1)`**. Khuôn "cận dưới" quen thuộc.

### Cách 4 — `bisect_left` với `key` (Python 3.10+)

**Giải thích.** Dãy `[citations[i] ≥ n − i for i in range(n)]` là `False…False True…True` (không giảm), nên `bisect_left(range(n), True, key=lambda i: citations[i] ≥ n − i)` trả về `i` đầu tiên thỏa (hoặc `n`). Đáp án `n − i`. Ngắn gọn, dùng thư viện chuẩn.

In [13]:
def hindex2_bisect(citations: list[int]) -> int:
    n = len(citations)
    i = bisect_left(range(n), True, key=lambda i: citations[i] >= n - i)
    return n - i


for citations, expected in examples:
    print(f"citations={citations} -> {hindex2_bisect(citations)} (kỳ vọng {expected})")

citations=[0, 1, 3, 5, 6] -> 3 (kỳ vọng 3)
citations=[1, 2, 100] -> 2 (kỳ vọng 2)


**Độ phức tạp Cách 4:** **Time `O(log n)`** (số lần gọi `key` là `O(log n)`; `range` không tạo list), **Space `O(1)`**.

### Cách 5 — Bỏ qua tính đã sắp: đếm thùng như bài 274 (`O(n)`)

**Giải thích.** Để so sánh: dùng đếm thùng chặn ở `n` (cách của bài 274). Đúng nhưng `O(n)` thời gian và `O(n)` bộ nhớ, **không** tận dụng việc mảng đã sắp và không đạt yêu cầu logarit.

In [14]:
def hindex2_counting(citations: list[int]) -> int:
    n = len(citations)
    bucket = [0] * (n + 1)
    for c in citations:
        bucket[min(c, n)] += 1
    total = 0
    for h in range(n, -1, -1):
        total += bucket[h]
        if total >= h:
            return h
    return 0


for citations, expected in examples:
    print(f"citations={citations} -> {hindex2_counting(citations)} (kỳ vọng {expected})")

citations=[0, 1, 3, 5, 6] -> 3 (kỳ vọng 3)
citations=[1, 2, 100] -> 2 (kỳ vọng 2)


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(n)`**.

> Cách 2 là cách nộp bài (khớp lời giải tham khảo). Cách 3 và 4 là hai cách viết khác của cùng ý tưởng. Cách 1 và 5 dùng làm chuẩn đối chiếu.

## 8. TRACE từng bước

Ta theo dõi Cách 2 (nhị phân trên `h`) cho `[1, 2, 100]` và Cách 3 (nhị phân trên chỉ số) cho `[0, 1, 3, 5, 6]`.

In [15]:
def trace_binary_h(citations):
    n = len(citations)
    left, right = 0, n
    print(f"citations={citations}, n={n}")
    step = 0
    while left < right:
        step += 1
        mid = (left + right + 1) >> 1
        ok = citations[n - mid] >= mid
        print(f"  bước {step}: [{left}, {right}], mid={mid}: citations[{n - mid}]={citations[n - mid]} {'>=' if ok else '<'} {mid} -> {'left = mid' if ok else 'right = mid - 1'}")
        if ok:
            left = mid
        else:
            right = mid - 1
    print("  => h-index =", left)
    return left


assert trace_binary_h([1, 2, 100]) == 2
print()
assert trace_binary_h([0, 0, 0]) == 0

citations=[1, 2, 100], n=3
  bước 1: [0, 3], mid=2: citations[1]=2 >= 2 -> left = mid
  bước 2: [2, 3], mid=3: citations[0]=1 < 3 -> right = mid - 1
  => h-index = 2

citations=[0, 0, 0], n=3
  bước 1: [0, 3], mid=2: citations[1]=0 < 2 -> right = mid - 1
  bước 2: [0, 1], mid=1: citations[2]=0 < 1 -> right = mid - 1
  => h-index = 0


In [16]:
def trace_binary_index(citations):
    n = len(citations)
    lo, hi = 0, n
    print(f"citations={citations}, n={n}")
    while lo < hi:
        mid = (lo + hi) // 2
        ok = citations[mid] >= n - mid
        print(f"  [{lo}, {hi}), mid={mid}: c[{mid}]={citations[mid]} {'>=' if ok else '<'} n - mid = {n - mid} -> {'hi = mid' if ok else 'lo = mid + 1'}")
        if ok:
            hi = mid
        else:
            lo = mid + 1
    print(f"  => i = {lo}, h = n - i = {n - lo}")
    return n - lo


assert trace_binary_index([0, 1, 3, 5, 6]) == 3

citations=[0, 1, 3, 5, 6], n=5
  [0, 5), mid=2: c[2]=3 >= n - mid = 3 -> hi = mid
  [0, 2), mid=1: c[1]=1 < n - mid = 4 -> lo = mid + 1
  => i = 2, h = n - i = 3


Bảng tóm tắt (Cách 2, `[1, 2, 100]`, `n = 3`):

| bước | `[left, right]` | `mid` | `citations[n − mid]` | so với `mid` | cập nhật |
|---|---|---|---|---|---|
| 1 | `[0, 3]` | 2 | `citations[1] = 2` | `2 ≥ 2` | `left = 2` |
| 2 | `[2, 3]` | 3 | `citations[0] = 1` | `1 < 3` | `right = 2` |

Dừng khi `left = right = 2`: h-index `2`. Chỉ đọc 2 phần tử của mảng.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (nhị phân trên `h`). Đúng class `Solution`, đúng method `hIndex(self, citations: List[int]) -> int`.

In [17]:
from typing import List


class Solution:
    def hIndex(self, citations: List[int]) -> int:
        n = len(citations)
        left, right = 0, n                          # h nằm trong [0, n]
        while left < right:
            mid = (left + right + 1) >> 1           # làm tròn lên để luôn thu hẹp khi left = mid
            if citations[n - mid] >= mid:           # h bài cuối đều có >= mid trích dẫn
                left = mid
            else:
                right = mid - 1
        return left


for citations, expected in examples:
    assert Solution().hIndex(citations) == expected
print("PASS ví dụ đề cho class Solution")

PASS ví dụ đề cho class Solution


## 10. Vì sao thuật toán đúng

- **Điều kiện cho `h ≥ 1`:** vì mảng sắp tăng, `h` bài cuối là `h` bài trích dẫn nhiều nhất; chúng đều `≥ h` khi và chỉ khi phần tử nhỏ nhất `citations[n − h] ≥ h`. Nếu điều kiện sai thì không có `h` bài nào cùng `≥ h` (chọn bất kỳ `h` bài nào cũng chứa một bài không nằm trong `h` bài cuối, có trích dẫn `≤ citations[n − h] < h`; đúng hơn: số bài có `≥ h` trích dẫn nhiều nhất là các bài từ chỉ số `n − h + 1`, tức tối đa `h − 1` bài).
- **Đơn điệu:** nếu `P(h)` đúng thì `P(h')` đúng với `h' < h` (chứng minh ở mục 6), `P(0)` đúng theo quy ước. Tập `True` là một đoạn đầu `[0, H]` và `H` là h-index.
- **Tìm nhị phân đúng:** bất biến `P(left)` đúng và `P(right + 1)` sai (hoặc `right = n`). Với `mid = (left + right + 1) // 2 ∈ (left, right]`: nếu `P(mid)` đúng thì `left = mid` giữ `P(left)` đúng; nếu sai thì `right = mid − 1` giữ `P(right + 1)` sai. Khoảng thu hẹp nên dừng với `left = right = H`.
- **Chỉ số hợp lệ:** trong vòng lặp `mid ≥ left + 1 ≥ 1` và `mid ≤ right ≤ n` nên `n − mid ∈ [0, n − 1]`.
- **Cách 3 và 4** tìm `i` nhỏ nhất thỏa `citations[i] ≥ n − i`: dãy điều kiện đơn điệu `False…True…` vì `citations[i]` không giảm còn `n − i` giảm; đáp án `h = n − i` (hoặc `0` nếu không có `i`).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(citations)`.

- **Cách 1 (quét tuyến tính):** Time `O(n)`, Space `O(1)`.
- **Cách 2 (nhị phân trên `h`):** Time **`O(log n)`**, Space `O(1)`.
- **Cách 3 (nhị phân trên chỉ số):** Time `O(log n)`, Space `O(1)`.
- **Cách 4 (`bisect_left` với `key`):** Time `O(log n)`, Space `O(1)`.
- **Cách 5 (đếm thùng, bỏ qua tính sắp):** Time `O(n)`, Space `O(n)`.
- Số bước nhị phân tối đa `⌈log₂(n + 1)⌉ ≈ 17` với `n = 10⁵`. Mỗi bước `O(1)`.

## 12. Bộ test

Ta gom các cách vào từ điển `impls`. `oracle` thử mọi `h` từ `0` đến `n + 1` bằng cách đếm trực tiếp số bài có `≥ h` trích dẫn (không dùng thứ tự, nhị phân hay thùng). Mảng ngẫu nhiên được **sắp tăng** trước khi đưa vào (đúng như đề).

In [18]:
sol = Solution()
impls = {
    "linear": hindex2_linear,
    "binary_h": hindex2_binary_h,
    "binary_index": hindex2_binary_index,
    "bisect": hindex2_bisect,
    "counting": hindex2_counting,
    "submit": sol.hIndex,
}


def oracle(citations):
    best = 0
    for h in range(0, len(citations) + 2):
        if sum(1 for c in citations if c >= h) >= h:
            best = h
    return best


def run_all(citations):
    out = {}
    for name, fn in impls.items():
        original = list(citations)
        out[name] = fn(original)
        assert original == citations, (name, "đã sửa đầu vào")
    return out


for citations, expected in examples:
    assert oracle(citations) == expected
    for name, got in run_all(citations).items():
        assert got == expected, (name, citations, got, expected)
print("PASS ví dụ của đề (mọi cách và oracle độc lập)")

PASS ví dụ của đề (mọi cách và oracle độc lập)


### 12.1. Các trường hợp biên

In [19]:
def check_all(citations, expected, label):
    assert citations == sorted(citations), "đầu vào phải sắp tăng"
    want = oracle(citations) if len(citations) <= 2000 else hindex2_counting(citations)    # oracle O(n^2) chỉ dùng cho n nhỏ
    assert want == expected, (label, "oracle", want, expected)
    for name, got in run_all(citations).items():
        assert got == expected, (label, name, got, expected)
    print("PASS", label, "->", expected)


check_all([0], 0, "n = 1, không có trích dẫn")
check_all([1], 1, "n = 1, một trích dẫn")
check_all([1000], 1, "n = 1, trích dẫn lớn")
check_all([0, 0, 0, 0], 0, "toàn 0")
check_all([1, 1, 1, 1], 1, "toàn 1")
check_all([4, 4, 4, 4], 4, "toàn n: h = n")
check_all([100, 100, 100], 3, "rất lớn: h = n")
check_all([0, 1, 3, 5, 6], 3, "ví dụ 1")
check_all([1, 2, 100], 2, "ví dụ 2")
check_all([0, 1], 1, "n = 2, h = 1")
check_all([2, 2], 2, "n = 2, h = 2")
check_all([0, 5], 1, "một bài lớn, một bài 0")
check_all([3, 4, 5, 8, 10], 4, "[3,4,5,8,10]: h = 4")
check_all([1000] * 100000, 1000, "n = 10^5 toàn 1000: h = 1000")
check_all(sorted(list(range(1001)) * 100), 991, "n = 100100 (0..1000 lặp 100 lần): h = 991")

PASS n = 1, không có trích dẫn -> 0
PASS n = 1, một trích dẫn -> 1
PASS n = 1, trích dẫn lớn -> 1
PASS toàn 0 -> 0
PASS toàn 1 -> 1
PASS toàn n: h = n -> 4
PASS rất lớn: h = n -> 3
PASS ví dụ 1 -> 3
PASS ví dụ 2 -> 2
PASS n = 2, h = 1 -> 1
PASS n = 2, h = 2 -> 2
PASS một bài lớn, một bài 0 -> 1
PASS [3,4,5,8,10]: h = 4 -> 4
PASS n = 10^5 toàn 1000: h = 1000 -> 1000
PASS n = 100100 (0..1000 lặp 100 lần): h = 991 -> 991


### 12.2. Tính chất và phản ví dụ

In [20]:
random.seed(275)

# Tính chất 1: 0 <= h <= min(n, max); N(h) >= h; N(h + 1) < h + 1
for _ in range(500):
    n = random.randint(1, 40)
    citations = sorted(random.randint(0, 50) for _ in range(n))
    h = sol.hIndex(citations)
    assert 0 <= h <= min(n, citations[-1])
    assert sum(c >= h for c in citations) >= h
    assert sum(c >= h + 1 for c in citations) < h + 1
print("PASS tính chất: N(h) >= h và N(h + 1) < h + 1")

# Tính chất 2: số lần đọc phần tử của mảng <= ceil(log2(n + 1)) + 1 (đếm bằng lớp bọc)
class Counted(list):
    reads = 0
    def __getitem__(self, i):
        Counted.reads += 1
        return super().__getitem__(i)

for n in (1, 10, 1000, 10**5):
    arr = Counted(sorted(random.randint(0, 1000) for _ in range(n)))
    Counted.reads = 0
    sol.hIndex(arr)
    bound = (n + 1).bit_length() + 1
    print(f"n={n}: số lần đọc = {Counted.reads} <= {bound}")
    assert Counted.reads <= bound

# Phản ví dụ: điều kiện ">" thay cho ">="
def sai_lon_hon(citations):
    n = len(citations)
    left, right = 0, n
    while left < right:
        mid = (left + right + 1) >> 1
        if citations[n - mid] > mid:                  # sai: phải là >=
            left = mid
        else:
            right = mid - 1
    return left

print("[1,2,100] -> sai:", sai_lon_hon([1, 2, 100]), "| đúng:", sol.hIndex([1, 2, 100]))
assert sai_lon_hon([1, 2, 100]) != 2

PASS tính chất: N(h) >= h và N(h + 1) < h + 1
n=1: số lần đọc = 1 <= 3
n=10: số lần đọc = 4 <= 5
n=1000: số lần đọc = 10 <= 11


n=100000: số lần đọc = 16 <= 18
[1,2,100] -> sai: 1 | đúng: 2


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [21]:
random.seed(2026)
for trial in range(3000):
    n = random.randint(1, 25)
    hi = random.choice([1, 3, 10, 30, 1000])
    citations = sorted(random.randint(0, hi) for _ in range(n))
    expected = oracle(citations)
    for name, got in run_all(citations).items():
        assert got == expected, (name, citations, got, expected)

for trial in range(30):                          # n tới 10^5
    n = random.randint(20000, 100000)
    citations = sorted(random.randint(0, random.choice([5, 50, 1000])) for _ in range(n))
    expected = hindex2_counting(citations)
    for name, fn in impls.items():
        assert fn(list(citations)) == expected, (name, n)
print("PASS 3000 input ngẫu nhiên khớp oracle độc lập; 30 mảng lớn (n tới 10^5) khớp giữa các cách")

PASS 3000 input ngẫu nhiên khớp oracle độc lập; 30 mảng lớn (n tới 10^5) khớp giữa các cách


## 13. Các lỗi người mới hay gặp

### Lỗi 1: `mid = (left + right) // 2` cùng `left = mid` gây vòng lặp vô hạn

In [22]:
def sai_vo_han(citations, max_steps=100):
    n = len(citations)
    left, right = 0, n
    steps = 0
    while left < right and steps < max_steps:       # chặn bước để minh họa, thực tế sẽ treo
        mid = (left + right) // 2                    # sai: làm tròn xuống
        if citations[n - mid] >= mid:
            left = mid                               # khi right = left + 1: mid = left, không thu hẹp
        else:
            right = mid - 1
        steps += 1
    return left, steps


print("[1,2,100] -> (kết quả, số bước):", sai_vo_han([1, 2, 100]), "| đúng:", Solution().hIndex([1, 2, 100]))

[1,2,100] -> (kết quả, số bước): (2, 100) | đúng: 2


**Vì sao sai:** khi `right = left + 1`, `mid = (left + right) // 2 = left`, nên `left = mid` **không đổi**: vòng lặp chạy mãi (ở đây dừng nhờ giới hạn 100 bước). Khi cập nhật `left = mid` phải làm tròn **lên** (`+ 1`).

### Lỗi 2: Trả về chỉ số thay vì `h = n − i`

In [23]:
def sai_tra_chi_so(citations):
    n = len(citations)
    lo, hi = 0, n
    while lo < hi:
        mid = (lo + hi) // 2
        if citations[mid] >= n - mid:
            hi = mid
        else:
            lo = mid + 1
    return lo                                       # sai: lo là chỉ số i, h = n - i


print("[0,1,3,5,6] -> sai:", sai_tra_chi_so([0, 1, 3, 5, 6]), "| đúng:", Solution().hIndex([0, 1, 3, 5, 6]))

[0,1,3,5,6] -> sai: 2 | đúng: 3


**Vì sao sai:** tìm nhị phân trên chỉ số cho `i` (vị trí bắt đầu của `h` bài cuối). h-index là **số bài** `h = n − i`. Nhớ đổi lại trước khi trả.

### Lỗi 3: Lệch chỉ số `n − mid` và `n − mid − 1`

In [24]:
def sai_lech_chi_so(citations):
    n = len(citations)
    left, right = 0, n
    while left < right:
        mid = (left + right + 1) >> 1
        if citations[n - mid - 1] >= mid:             # sai: bài thứ mid từ cuối là citations[n - mid]
            left = mid
        else:
            right = mid - 1
    return left


print("[1,2,100] -> sai:", sai_lech_chi_so([1, 2, 100]), "| đúng:", Solution().hIndex([1, 2, 100]))

[1,2,100] -> sai: 1 | đúng: 2


**Vì sao sai:** `h` bài cuối bắt đầu ở chỉ số `n − h` (không phải `n − h − 1`). Ngoài ra khi `mid = n`, `n − mid − 1 = −1` và Python đọc **phần tử cuối** (chỉ số âm) thay vì báo lỗi, nên lỗi có thể bị che giấu ở một số input. Hãy tự kiểm tra bằng ví dụ nhỏ.

### Lỗi 4: Quên mảng chỉ cần `≥` (dùng `>`) hoặc quên `h = 0`

In [25]:
print("[1,2,100] -> dùng '>' ra:", sai_lon_hon([1, 2, 100]), "| đúng:", Solution().hIndex([1, 2, 100]))
print("[0,0,0] -> đúng:", Solution().hIndex([0, 0, 0]), "(h = 0, vòng lặp không đọc mảng ngoài phạm vi)")

[1,2,100] -> dùng '>' ra: 1 | đúng: 2
[0,0,0] -> đúng: 0 (h = 0, vòng lặp không đọc mảng ngoài phạm vi)


**Vì sao sai:** đề yêu cầu **ít nhất `h` trích dẫn** (`≥ h`), nên bài hạng `h` có đúng `h` trích dẫn vẫn tính. Dùng `>` loại nhầm trường hợp bằng nhau. Khoảng tìm kiếm bắt đầu từ `left = 0` nên `h = 0` (không có bài nào đạt) được bao gồm.

## 14. Mẹo làm nhanh & nhận dạng pattern

- "Mảng **đã sắp**, cần `O(log n)`": nghĩ ngay **tìm nhị phân**; xác định điều kiện **đơn điệu** `True…True False…False` (hoặc ngược lại).
- h-index trên mảng sắp tăng: điều kiện `citations[n − h] ≥ h`; tìm **`h` lớn nhất** thỏa (khuôn "True cuối cùng").
- Chọn `mid`: **làm tròn lên** khi cập nhật `lo = mid`; **làm tròn xuống** khi cập nhật `hi = mid`.
- Đổi biến (chỉ số `i = n − h`) có thể biến "lớn nhất" thành "nhỏ nhất" và dùng `bisect_left`.
- Kiểm tra các biên: `h = 0`, `h = n`, mảng toàn `0`, toàn lớn.
- Test độ phức tạp bằng cách **đếm số lần đọc phần tử** (lớp bọc `list` đếm `__getitem__`).

## 15. Biến thể của bài

1. **H-Index (LeetCode 274):** mảng chưa sắp, `O(n)` bằng đếm thùng.
2. **Tìm `i` đầu tiên thỏa một điều kiện đơn điệu** (Binary Search tổng quát, LeetCode 34, 278).
3. **Giá trị lớn nhất `h` sao cho `h` phần tử cuối có tổng ≥ `h²`** (g-index trên mảng sắp).
4. **H-index khi mảng sắp giảm dần:** đảo chỉ số.
5. **Nhiều truy vấn "thêm `k` trích dẫn":** nhị phân với hàm chi phí.

In [26]:
def hindex_sorted_desc(citations):
    """Biến thể 4: mảng sắp GIẢM dần: bài hạng h là citations[h - 1]; điều kiện c[h - 1] >= h là True...True False...False theo h."""
    n = len(citations)
    lo, hi = 0, n
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if citations[mid - 1] >= mid:
            lo = mid
        else:
            hi = mid - 1
    return lo


def g_index_sorted_asc(citations):
    """Biến thể 3 (đơn giản): g lớn nhất sao cho g bài cuối có tổng >= g^2; quét vì tổng không đơn điệu rõ ràng theo g."""
    n = len(citations)
    best, total = 0, 0
    for g in range(1, n + 1):
        total += citations[n - g]
        if total >= g * g:
            best = g
    return best


def first_index_ge(a, x):
    """Biến thể 2: chỉ số đầu tiên có a[i] >= x (bisect_left)."""
    return bisect_left(a, x)


random.seed(15)
for _ in range(300):
    asc = sorted(random.randint(0, 30) for _ in range(random.randint(1, 20)))
    desc = asc[::-1]
    assert hindex_sorted_desc(desc) == Solution().hIndex(asc)
    x = random.randint(0, 31)
    assert first_index_ge(asc, x) == next((i for i, v in enumerate(asc) if v >= x), len(asc))
assert g_index_sorted_asc([0, 1, 3, 5, 6]) == 3
print("PASS biến thể: sắp giảm dần, tìm cận dưới")

PASS biến thể: sắp giảm dần, tìm cận dưới


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét tuyến tính | `i` đầu tiên có `c[i] ≥ n − i` | `O(n)` | `O(1)` | hiểu đề; oracle |
| 2. Nhị phân trên `h` | `c[n − mid] ≥ mid`, `mid` làm tròn lên | **`O(log n)`** | `O(1)` | **cách nộp bài** |
| 3. Nhị phân trên chỉ số | cận dưới của `i`, đáp án `n − i` | `O(log n)` | `O(1)` | khuôn lower bound quen thuộc |
| 4. `bisect_left` với `key` | thư viện, `key=lambda i: c[i] ≥ n − i` | `O(log n)` | `O(1)` | ngắn gọn (Python ≥ 3.10) |
| 5. Đếm thùng (bài 274) | bỏ qua tính sắp | `O(n)` | `O(n)` | khi mảng chưa sắp |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `(0 + 5 + 1) // 2`, `[0, 1, 3, 5, 6][5 - 3]`, `Solution().hIndex([0])`, `Solution().hIndex([5, 5, 5])`, `Solution().hIndex([0, 0, 4, 4])`.

**Bài 2 (tính tay).** Với `citations = [0, 2, 2, 3, 7, 9]`: lập bảng `i`, `c[i]`, `n − i`, `c[i] ≥ n − i` và tìm h-index; sau đó mô phỏng nhị phân trên `h` (ghi `left`, `right`, `mid`).

**Bài 3 (điền TODO).** Hoàn thành `hindex2_todo(citations)` theo Cách 2.

In [27]:
def hindex2_todo(citations):
    n = len(citations)
    left, right = 0, n
    # TODO 1: trong khi left < right
    #   TODO 2: mid = (left + right + 1) >> 1
    #   TODO 3: nếu citations[n - mid] >= mid thì left = mid, ngược lại right = mid - 1
    # TODO 4: trả về left
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `hindex2_lower_bound(citations)` theo Cách 3 (nhị phân trên chỉ số, `h = n − i`). Kiểm tra bằng hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_at_least(citations, x)` trả về số bài có **≥ x** trích dẫn trong mảng **đã sắp tăng** bằng `bisect_left` (`O(log n)`). Dùng nó để viết lại h-index bằng nhị phân trên `h`. (b) Viết `hindex_with_extra_paper(citations, c)` trả về h-index sau khi **thêm một bài mới** có `c` trích dẫn (mảng sắp tăng), trong `O(log n)` (dùng `bisect`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `(left + right + 1) // 2` làm tròn lên; `h` bài cuối bắt đầu ở chỉ số `n − h`. Bài 2: kiểm tra phần tử nhỏ nhất của `h` bài cuối.
- **Mức 2 (vừa):** Bài 1: `3`, `3`, `0`, `3`, `2`. Bài 2: `[0,2,2,3,7,9]`, `n = 6`: `i=0: 0≥6? không`; `1: 2≥5 không`; `2: 2≥4 không`; `3: 3≥3 có` → `h = 3`. Nhị phân: `[0,6]` mid=3: `c[3]=3≥3` → left=3; `[3,6]` mid=5: `c[1]=2<5` → right=4; `[3,4]` mid=4: `c[2]=2<4` → right=3; kết thúc `3`. Bài 3: bốn TODO trong comment.
- **Mức 3 (sâu):** Bài 4: `lo, hi = 0, n`; `mid = (lo + hi) // 2`; nếu `c[mid] ≥ n − mid` thì `hi = mid` ngược lại `lo = mid + 1`; trả `n − lo`. Bài 5(a): `count_at_least = n − bisect_left(c, x)`; điều kiện `h` hợp lệ: `count_at_least(c, h) ≥ h`. Bài 5(b): chèn `c` bằng `bisect.insort` rồi gọi Cách 2 (sao chép mảng nên `O(n)`), hoặc suy luận cục bộ: thêm bài mới tăng h-index nhiều nhất `1`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
from bisect import insort

# Bài 1
assert (0 + 5 + 1) // 2 == 3 and [0, 1, 3, 5, 6][5 - 3] == 3
assert Solution().hIndex([0]) == 0 and Solution().hIndex([5, 5, 5]) == 3 and Solution().hIndex([0, 0, 4, 4]) == 2

# Bài 2
assert [(i, v >= 6 - i) for i, v in enumerate([0, 2, 2, 3, 7, 9])] == [(0, False), (1, False), (2, False), (3, True), (4, True), (5, True)]
assert Solution().hIndex([0, 2, 2, 3, 7, 9]) == 3

# Bài 3
def hindex2_todo_ans(citations):
    n = len(citations)
    left, right = 0, n
    while left < right:
        mid = (left + right + 1) >> 1
        if citations[n - mid] >= mid:
            left = mid
        else:
            right = mid - 1
    return left

for citations, expected in examples:
    assert hindex2_todo_ans(citations) == expected

# Bài 4
def hindex2_lower_bound(citations):
    n = len(citations)
    lo, hi = 0, n
    while lo < hi:
        mid = (lo + hi) // 2
        if citations[mid] >= n - mid:
            hi = mid
        else:
            lo = mid + 1
    return n - lo

for citations, expected in examples:
    assert hindex2_lower_bound(citations) == expected

# Bài 5(a), 5(b)
def count_at_least(citations, x):
    return len(citations) - bisect_left(citations, x)

def hindex_via_count(citations):
    lo, hi = 0, len(citations)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if count_at_least(citations, mid) >= mid:
            lo = mid
        else:
            hi = mid - 1
    return lo

def hindex_with_extra_paper(citations, c):
    arr = citations[:]
    insort(arr, c)
    return Solution().hIndex(arr)

random.seed(19)
for _ in range(300):
    arr = sorted(random.randint(0, 25) for _ in range(random.randint(1, 15)))
    assert hindex_via_count(arr) == Solution().hIndex(arr)
    c = random.randint(0, 25)
    h2 = hindex_with_extra_paper(arr, c)
    assert Solution().hIndex(arr) <= h2 <= Solution().hIndex(arr) + 1
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (nhị phân trên `h`, mảng sắp tăng, điều kiện `citations[n − h] ≥ h`)

```python
class Solution:
    def hIndex(self, citations: List[int]) -> int:
        n = len(citations)
        left, right = 0, n
        while left < right:
            mid = (left + right + 1) >> 1
            if citations[n - mid] >= mid:
                left = mid
            else:
                right = mid - 1
        return left
```

**Checklist**

- [ ] Hiểu: `h` hợp lệ ⇔ phần tử nhỏ nhất của `h` bài cuối `citations[n − h] ≥ h`; điều kiện `True…True False…False` theo `h`?
- [ ] Khoảng tìm kiếm `h ∈ [0, n]` (bao gồm `0` và `n`)?
- [ ] `mid = (left + right + 1) >> 1` (làm tròn **lên**) khi cập nhật `left = mid`?
- [ ] Dùng `>=` (ít nhất `h` trích dẫn), không dùng `>`?
- [ ] Chỉ số đúng: `citations[n − mid]` (không phải `n − mid − 1`), luôn trong `[0, n − 1]`?
- [ ] Đã thử: `[0]`, `[1000]`, toàn `0`, toàn lớn, `n = 10⁵`?
- [ ] Đã đối chiếu với brute force đếm trực tiếp mọi `h` trên mảng sắp tăng ngẫu nhiên và kiểm tra số lần đọc `O(log n)`?
- [ ] Skeleton đúng `class Solution` / `def hIndex(self, citations: List[int]) -> int`?